In [1]:
import os
print("Folders currently in your input directory:")
try:
    print(os.listdir('/kaggle/input'))
except Exception as e:
    print("Error:", e)

Folders currently in your input directory:
['notebooks', 'datasets']


In [2]:
import os
import shutil
import subprocess

# 1. Reset working directory to base to prevent nested clones
os.chdir('/kaggle/working')

repo_url = "https://github.com/Naveentijo/ChestDiseaseAI.git"
repo_name = "ChestDiseaseAI"

# Clean up previous nested folders if they exist
if os.path.exists(repo_name) and os.path.exists(os.path.join(repo_name, repo_name)):
    print("Detected nested folders. Cleaning directory...")
    shutil.rmtree(repo_name)

# 2. Clone or pull the repository
if not os.path.exists(repo_name):
    print("Cloning repository from GitHub...")
    subprocess.run(["git", "clone", repo_url])
else:
    print("Repository exists. Pulling latest updates...")
    os.chdir(repo_name)
    subprocess.run(["git", "pull"])
    os.chdir('/kaggle/working')

# 3. Enter the clean repository directory
os.chdir(f"/kaggle/working/{repo_name}")
print(f"Current working directory set to: {os.getcwd()}")

# 4. Install dependencies
print("Installing Python dependencies...")
subprocess.run(["pip", "install", "-r", "requirements.txt"])

# ========================================================
# 5. DEPTH-LIMITED SEARCH FOR CHEXPERT DATASET
# ========================================================
def find_dataset_folder(base_path, target_file="train.csv", max_depth=4, current_depth=0):
    """Searches directories only (no files) to find target folder instantly."""
    if current_depth > max_depth:
        return None
    try:
        contents = os.listdir(base_path)
    except Exception:
        return None
        
    if target_file in contents:
        return base_path
        
    for item in contents:
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            found = find_dataset_folder(item_path, target_file, max_depth, current_depth + 1)
            if found:
                return found
    return None

def print_input_structure(path, max_depth=3, current_depth=0):
    """Prints directory tree of mounted inputs (ignoring images) for debugging."""
    if current_depth > max_depth:
        return
    try:
        items = os.listdir(path)
    except Exception:
        return
    for item in items:
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            print("  " * current_depth + f"📁 {item}/")
            print_input_structure(item_path, max_depth, current_depth + 1)
        elif item.endswith(".csv"):
            print("  " * current_depth + f"📄 {item}")

kaggle_input_dir = "/kaggle/input"
dataset_found = find_dataset_folder(kaggle_input_dir, "train.csv")

if dataset_found:
    print(f"\nSuccess: CheXpert dataset found at: {dataset_found}")
else:
    print("\n[ERROR] CheXpert dataset folder containing train.csv not found.")
    print("Showing structure of your /kaggle/input directory:")
    print_input_structure(kaggle_input_dir)
    raise RuntimeError(
        "CheXpert dataset not found. Please verify you have added the "
        "dataset (not a notebook) from the sidebar."
    )

# ========================================================
# 6. DYNAMICALLY RESOLVE RELATIVE PATHS
# ========================================================
train_csv_abs = os.path.join(dataset_found, "train.csv")
valid_csv_abs = os.path.join(dataset_found, "valid.csv")

# Calculate relative paths to override default config settings
rel_train_path = os.path.relpath(train_csv_abs, dataset_found)
rel_valid_path = os.path.relpath(valid_csv_abs, dataset_found)

print(f"Located train metadata: {rel_train_path}")
print(f"Located valid metadata: {rel_valid_path}")

# ========================================================
# 7. RUN PYTORCH TRAINING ON GPU
# ========================================================
print("Launching training loop on GPU...")
os.environ["CHEST_AI_DEVICE"] = "cuda"
os.environ["PYTHONPATH"] = os.getcwd()

# Run training
subprocess.run([
    "python", "-u", "ml/chest_ai/train.py", 
    "--data-dir", dataset_found,
    "--csv-train-path", rel_train_path,
    "--csv-valid-path", rel_valid_path,
    "--epochs", "15", 
    "--batch-size", "32"
])

Cloning repository from GitHub...


Cloning into 'ChestDiseaseAI'...


Current working directory set to: /kaggle/working/ChestDiseaseAI
Installing Python dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.



Success: CheXpert dataset found at: /kaggle/input/datasets/ashery/chexpert
Located train metadata: train.csv
Located valid metadata: valid.csv
Launching training loop on GPU...


[2026-07-10 08:59:39] INFO [chest_ai:train.py:66] - Setting up dataloaders...
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
[2026-07-10 08:59:40] INFO [chest_ai:model.py:25] - Initializing ChestClassifier with backbone: densenet121 (pretrained=True)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 143MB/s]
[2026-07-10 08:59:41] INFO [chest_ai:trainer.py:56] - AMP (Mixed Precision) enabled for CUDA training.
[2026-07-10 08:59:41] INFO [chest_ai:trainer.py:63] - Initialized TensorBoard writer at: ./ml/logs/tensorboard_20260710-085941
[2026-07-10 08:59:41] INFO [chest_ai:trainer.py:76] - Starting training on device: cuda for 15 epochs.
Validating: 100%|██████████| 8/8 [00:04<00:00,  1.74it/s]
[2026-07-10 10:16:24] INFO [chest_ai:trainer.py:92] - Epoch 01/15 | Train Loss: 0.9313 | Val Loss: 1.2204 | Val Macro AUROC: 0.8716 | Time: 4603.4s
[2026-07-10 10:16:24] INFO [chest_ai:trainer.py:124] - Val Macro AUROC improved from -inf to 0.8716.
[2026-07-10 10:16:24] INFO [chest_ai:checkpoint.py:51] - Saved training checkpoint to: ./ml/checkpoints/checkpoint_epoch_001.pth
[2026-07-10 10:16:24] INFO [chest_ai:checkpoint.py:57] - New best model identified. Saved copy to: ./ml/checkpoints/best_model.pth
Validating: 100%|██████████| 8/8 [00:02<00:00,  

CompletedProcess(args=['python', '-u', 'ml/chest_ai/train.py', '--data-dir', '/kaggle/input/datasets/ashery/chexpert', '--csv-train-path', 'train.csv', '--csv-valid-path', 'valid.csv', '--epochs', '15', '--batch-size', '32'], returncode=0)